# Content Popularity Tracker
**Company:** Atlassian (GothamLoop question bank) · **Category:** Coding · **Tags:** Live Screen, Onsite Loop, Hash Tables, Heaps · **Difficulty/Frequency:** Very Common (8/10)


## Concepts

**What this problem is really testing:**
- Bucketing items into groups by their current value
- Amortized complexity (why a loop that "looks slow" can still be cheap overall)
- A heap-based idea that seems reasonable but turns out to be flawed here

**Why each one shows up here:**
- This is a "keep track of the running maximum while values go up and down by 1" problem — the same shape as LeetCode's *All O`one Data Structure*.
- A plain hash map gives you O(1) updates, but then finding the max means scanning everything — O(n).
- A heap looks like the fix, but Python's `heapq` can't update an entry's priority in place — so you're back to scanning, just in a different disguise.
- The real fix uses a fact this problem gives you for free: **scores only ever change by exactly 1**. That means you can group ("bucket") content IDs by their current score, and just track which bucket is currently the highest.

**The one idea to hold onto:** don't find the max by searching for it. Keep the answer updated as a side effect of every change, and only do extra work in proportion to how far the answer actually needs to move.

---

### Quick primers — the building blocks used below

**What is a Hash Map?**
- A hash map (Python `dict`) stores key → value pairs by hashing the key to a slot — giving **O(1) average** insert/lookup/delete.
- **In Python:** use `dict` for key→value. Use `set` when you only need fast membership + removal — exactly what "the group of IDs currently at score s" needs.

**What is a Heap (priority queue)?**
- A binary heap keeps the min (or max) element accessible in O(1), with O(log n) push/pop, by keeping one rule true: "parent ≤ children".
- **The catch, specifically for this problem:** a normal heap has no fast (O(log n)) way to change one element's priority, or remove an item that isn't at the top.
- If a score can go down, your options are: rebuild the whole heap, or push a new entry and later throw away the old, now-stale one when it resurfaces (**lazy deletion**) — which can force `mostPopular()` to pop through several stale entries in a row.

**Amortized analysis (why a "slow-looking" loop can still be cheap).**
- A single operation can *look* expensive in the worst case — e.g. a `while` loop that walks through several values.
- But if you can show the **total** work across *every* call combined is bounded, then the *average* cost per call is small — even though no individual call is guaranteed to be fast.
- This is different from "best case" or "worst case" per call — it's a statement about a whole sequence of operations, not any one of them.

**Bucketing by value.**
- When values change in small, predictable steps (here, always ±1), you can group items into "buckets" keyed by their current value.
- Moving an item when its value changes becomes: remove from one bucket, add to another — both **O(1)** — instead of searching for anything.


## Problem Statement

Design `ContentPopularity` with:
- `increasePopularity(contentId)` -- +1 (thumbs up).
- `decreasePopularity(contentId)` -- -1 (thumbs down).
- `mostPopular()` -- the content ID with the highest popularity (any one, on ties); `-1` if nothing exists.

**Official follow-ups (covered below):** non-existent ID on decrease; top-K extension; scores that change by more than 1; removing content entirely; deterministic tie-breaking.


### Approach 1 -- Naive (hash map + linear scan)

**Idea:** a plain `dict` from content ID to score. `increase`/`decrease` are O(1) dict updates. `mostPopular()` scans every entry to find the max.

**Time complexity:** O(1) for `increase`/`decrease`; **O(n)** for `mostPopular()`, where n is the number of distinct IDs.

**Space complexity:** O(n).


In [ ]:
from typing import Dict


class ContentPopularityNaive:
    def __init__(self) -> None:
        self.score: Dict[int, int] = {}

    def increasePopularity(self, contentId: int) -> None:
        self.score[contentId] = self.score.get(contentId, 0) + 1

    def decreasePopularity(self, contentId: int) -> None:
        if contentId in self.score:
            self.score[contentId] -= 1

    def mostPopular(self) -> int:
        if not self.score:
            return -1
        return max(self.score, key=self.score.get)   # O(n) scan every call


### Approach 2 -- Heap with lazy deletion (why it *almost* works)

**Idea:** push `(-score, contentId)` onto a max-heap-via-negation whenever a score changes. `mostPopular()` peeks the top, but the top might be **stale** (that ID's score has since changed via a newer push) -- so keep popping until the top entry's stored score matches the ID's *current* score in a side `dict`.

**Time complexity:** each push is O(log n). `mostPopular()` is O(log n) **amortized** across all calls in the best case, but a single call can pop through an arbitrary run of stale entries first -- there's no per-call guarantee, only an amortized one tied to total pushes.

**Space complexity:** O(n + P) where P is the total number of pushes ever made -- stale entries accumulate in the heap until they happen to surface and get discarded.


In [ ]:
import heapq
from typing import Dict, List, Tuple


class ContentPopularityHeap:
    def __init__(self) -> None:
        self.score: Dict[int, int] = {}
        self.heap: List[Tuple[int, int]] = []   # (-score, contentId), possibly stale

    def _push(self, contentId: int) -> None:
        heapq.heappush(self.heap, (-self.score[contentId], contentId))

    def increasePopularity(self, contentId: int) -> None:
        self.score[contentId] = self.score.get(contentId, 0) + 1
        self._push(contentId)

    def decreasePopularity(self, contentId: int) -> None:
        if contentId not in self.score:
            return
        self.score[contentId] -= 1
        self._push(contentId)

    def mostPopular(self) -> int:
        while self.heap:
            neg_score, contentId = self.heap[0]
            if -neg_score == self.score.get(contentId):   # still current -- not stale
                return contentId
            heapq.heappop(self.heap)                       # stale entry, discard and keep looking
        return -1


### Approach 3 -- Optimal (score buckets + a max pointer)

**Idea:** maintain `score[id] -> current popularity`, `buckets[s] -> set of ids at score s`, and a `max_score` pointer. `increase`/`decrease` move an ID between two adjacent buckets -- pure set add/discard, O(1). If the *old* max bucket becomes empty on a decrease, walk `max_score` down until a non-empty bucket is found.

**Time complexity:** O(1) for `increase` and `mostPopular()`. Because `_move` deletes a bucket the instant it empties, `buckets` never contains gaps -- so whenever `decreasePopularity` needs to walk `max_score` down, `new_score` (exactly one below the old max) is *already* guaranteed non-empty (it just received the ID that moved). The walk-down is therefore **exactly one step, every time**, not merely "amortized O(1)" -- worst case O(1) too.

**Space complexity:** O(n) -- `score` holds one entry per ID, and every ID lives in exactly one bucket's set at a time.

> **A bug worth catching.** The source material's own reference answer bounds that walk-down loop with `while self.max_score > 0 and ...`. That extra `> 0` silently assumes scores never go negative -- but nothing in the problem statement guarantees that (a content ID can rack up more thumbs-down than thumbs-up). Once a lone ID's score crosses from 0 to -1, that guard stops the walk at `max_score = 0`, `mostPopular()` then finds bucket 0 empty and returns **-1** even though the ID is still being tracked. Since the walk only ever needs to move exactly one step (see above), the fix is simply to drop the artificial floor -- shown in the code below and caught by the "score goes negative" edge case in Verification.


In [ ]:
from typing import Dict, Set


class ContentPopularity:
    def __init__(self) -> None:
        self.score: Dict[int, int] = {}
        self.buckets: Dict[int, Set[int]] = {}
        self.max_score = 0

    def _move(self, contentId: int, old_score: int, new_score: int) -> None:
        if old_score in self.buckets:
            self.buckets[old_score].discard(contentId)
            if not self.buckets[old_score]:
                del self.buckets[old_score]           # keep buckets sparse -- only non-empty scores
        self.buckets.setdefault(new_score, set()).add(contentId)
        self.score[contentId] = new_score

    def increasePopularity(self, contentId: int) -> None:
        old_score = self.score.get(contentId, 0)
        new_score = old_score + 1
        self._move(contentId, old_score, new_score)
        if new_score > self.max_score:
            self.max_score = new_score

    def decreasePopularity(self, contentId: int) -> None:
        if contentId not in self.score:
            return
        old_score = self.score[contentId]
        new_score = old_score - 1
        self._move(contentId, old_score, new_score)
        if old_score == self.max_score and old_score not in self.buckets:
            # No floor at 0: scores CAN go negative (see the note below), and `new_score`
            # (old_score - 1) is always guaranteed non-empty -- it just received `contentId` --
            # so this loop is guaranteed to terminate there even without an artificial floor.
            while self.max_score not in self.buckets:
                self.max_score -= 1

    def mostPopular(self) -> int:
        if not self.buckets or self.max_score not in self.buckets:
            return -1
        return next(iter(self.buckets[self.max_score]))   # any ID in the max bucket, O(1)


## Verification

Run a scripted sequence against all three implementations and confirm they agree, plus check the edge cases each Talking Point calls out.

In [ ]:
def run_ops(cp, ops):
    """ops: list of ('inc'|'dec'|'top', contentId_or_None) -> list of mostPopular() results."""
    results = []
    for kind, arg in ops:
        if kind == "inc":
            cp.increasePopularity(arg)
        elif kind == "dec":
            cp.decreasePopularity(arg)
        else:
            results.append(cp.mostPopular())
    return results


ops = [
    ("top", None),          # -1, nothing exists yet
    ("inc", 1), ("inc", 1), ("inc", 2),
    ("top", None),          # 1 has score 2, the max
    ("inc", 2), ("inc", 2),
    ("top", None),          # 2 now has score 3, the max
    ("dec", 2), ("dec", 2),
    ("top", None),          # 2 back to 1, 1 has 2 -> max is 1
    ("dec", 1), ("dec", 1),
    ("top", None),          # 1 at 0, 2 at 1 -> max is 2
]
expected = [-1, 1, 2, 1, 2]

for cls in (ContentPopularityNaive, ContentPopularityHeap, ContentPopularity):
    got = run_ops(cls(), ops)
    assert got == expected, f"{cls.__name__} mismatch: {got} != {expected}"

# Edge cases
cp = ContentPopularity()
assert cp.mostPopular() == -1                    # nothing tracked yet
cp.decreasePopularity(99)                        # decrease on unknown id: documented no-op
assert cp.mostPopular() == -1
cp.increasePopularity(5)
assert cp.mostPopular() == 5
cp.decreasePopularity(5)
cp.decreasePopularity(5)                          # score goes negative; still the only id tracked
assert cp.mostPopular() == 5

# Tie handling: whichever id is returned must actually be at the max score
cp2 = ContentPopularity()
cp2.increasePopularity(10)
cp2.increasePopularity(20)
assert cp2.mostPopular() in (10, 20)
assert cp2.score[cp2.mostPopular()] == max(cp2.score.values())

print("All checks passed.")


## Discussion -- remaining follow-up directions

- **Top-K extension.** The bucket approach generalizes: to answer "top K IDs", you can't stop at just `max_score` -- walk buckets downward from `max_score`, collecting IDs until you have K (still efficient if K is small and scores are clustered near the top), or maintain a sorted structure over non-empty bucket keys (e.g. a small heap of bucket scores) for faster descent.
- **Scores that jump by more than 1 (a "super-like" worth +10).** The bucket structure itself still works -- `_move` doesn't care about the size of the jump. What breaks is the *amortized argument* for `decreasePopularity`'s walk-down: it relied on scores changing by exactly 1, so `max_score` only ever needs to step down one bucket at a time to find the next non-empty one. With arbitrary jumps, a big score could vacate a bucket far below any populated one, and you'd need a different structure (e.g. a sorted container of occupied scores, such as a heap of bucket keys or a balanced BST) to find the next-highest occupied score in better than O(range of scores).
- **Removing content entirely.** Add a `removeContent(contentId)` that deletes it from `score` and its current bucket, then re-runs the same "walk `max_score` down if its bucket is now empty" check as `decreasePopularity` -- shown below.
- **Deterministic tie-breaking (e.g. always the smallest ID).** Swap each bucket's `set` for a structure with a fast "give me the smallest" operation -- a small heap, or a sorted container -- which changes bucket operations from O(1) to O(log bucket size).


In [ ]:
class ContentPopularityWithRemoval(ContentPopularity):
    """Bonus: supports fully removing a content id from tracking."""

    def removeContent(self, contentId: int) -> None:
        if contentId not in self.score:
            return
        old_score = self.score.pop(contentId)
        self.buckets[old_score].discard(contentId)
        if not self.buckets[old_score]:
            del self.buckets[old_score]
        if old_score == self.max_score and old_score not in self.buckets:
            # Unlike decreasePopularity, removal doesn't guarantee a non-empty bucket just
            # below -- the removed id isn't moved anywhere -- so this walk can run all the
            # way out (everything removed). Guard on `self.buckets`, not an arbitrary floor.
            while self.buckets and self.max_score not in self.buckets:
                self.max_score -= 1
            if not self.buckets:
                self.max_score = 0


cp = ContentPopularityWithRemoval()
cp.increasePopularity(1)
cp.increasePopularity(2)
cp.increasePopularity(2)
assert cp.mostPopular() == 2
cp.removeContent(2)
assert cp.mostPopular() == 1
cp.removeContent(1)
assert cp.mostPopular() == -1          # everything removed
print("removeContent works as expected.")


## Empirical complexity check

The interesting comparison is Approach 1 (naive scan) vs. Approach 3 (buckets) under a workload that calls `mostPopular()` after *every* `increasePopularity()` -- the worst case for the naive scan. If the naive version is really O(n) per `mostPopular()` call, running n such pairs costs O(n^2) total; the bucket version should stay O(n) total.

| Growth when n doubles | Implies |
|---|---|
| ~2x | linear (bucket approach) |
| ~4x | quadratic (naive approach) |


In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark


def run_naive_workload(ids):
    cp = ContentPopularityNaive()
    for cid in ids:
        cp.increasePopularity(cid)
        cp.mostPopular()          # O(n) scan, called n times -> O(n^2) total


def run_bucket_workload(ids):
    cp = ContentPopularity()
    for cid in ids:
        cp.increasePopularity(cid)
        cp.mostPopular()          # O(1), called n times -> O(n) total


def make_worst_case(n):
    return (list(range(n)),)      # n distinct ids, each incremented once


solutions = {
    "naive scan (O(n) per call)": run_naive_workload,
    "buckets (O(1) per call)": run_bucket_workload,
}
sizes = [500, 1000, 2000, 4000]   # kept small: the naive side is quadratic
benchmark(solutions, make_worst_case, sizes, plot=True)


## 🧩 Patterns Learned

- **Bucket by value when updates are small, predictable steps.** Grouping items by their current value into a `dict[value] -> set[item]` turns "this item's value changed" into O(1) set moves -- the same idea behind bucket sort and counting sort.
- **Track a running extreme instead of recomputing it.** Maintaining `max_score` incrementally and only "walking it down" when its own bucket empties avoids ever scanning the full collection.
- **Amortized analysis: bound total work, not per-call work.** The walk-down loop in `decreasePopularity` looks like it could be O(n), but each score value is vacated at most once across the entire sequence of calls -- so total work is bounded, even though no single call has a tight worst-case bound on its own.
- **A heap isn't automatically the answer for "track the max under updates."** Heaps excel at "give me the extreme, then remove it" -- they don't support efficient arbitrary-key updates, which forces lazy deletion and reintroduces the scan you were trying to avoid.
- **State your policy for "operate on an ID that doesn't exist yet" explicitly.** No-op, initialize at zero, or raise -- any is defensible, but silently picking one without saying so is where interview points are lost.
- **Related problems:** LeetCode "All O`one Data Structure" (the direct analogue), LFU cache (bucket-by-frequency is the same trick), sliding window maximum (different technique, same "avoid recomputation" spirit).
- **Common pitfalls:** forgetting to delete empty bucket entries (leaks memory and breaks "is this score occupied" checks); assuming a heap gives O(log n) decrease-key for free; not deciding what "most popular" means when everything is tied or nothing has been added yet.
